# m_viz: MATPOWER case geovisualization
* conda: h-py312-basic

Parses MATPOWER `.m` case files (ACOPF or PF solutions) into normalized pandas DataFrames
(bus, gen, branch) and produces interactive Plotly geo maps. Geographic coordinates come from a
`.gic` or `.AUX` file. Generator fuel labels and multi-circuit branch data are read from the `.m`
file.

The map visualizes the solution stored in the `.m` file: bus loads, dispatched generation (marker
size proportional to MW), and branch loading percentage (viridis color scale).

Optionally, point `GRIDKIT_REPO` to a local GridKit repository clone to augment hover labels with
GridKit-assigned bus and branch IDs.

Key steps:
- **Colocated bus splitting**: buses sharing identical lat/lon are spread on a small circle so each is individually hoverable.
- **Generator fan-out**: when a bus has more than one generator, all generators at that bus are spread radially so each marker is individually hoverable; thin connector lines link them back to the bus. Optionally, single-generator buses can also be offset (`FAN_SINGLE_GEN = True`) to keep the bus marker visible underneath.
- **Fault bus overlay**: optionally highlights a faulted bus with a red marker.
- **Branch loading coloring**: branches colored by loading percentage when flow data is available.

Currently supported cases: **Hawaii40**, **Illinois (ACTIVSg200)**, **Texas (ACTIVSg2000)**, **WECC (ACTIVSg10k)**.


In [ ]:
from pathlib import Path
import importlib
import os
import sys

# py-utils is one level up from the notebooks/ directory
_py_utils = Path.cwd().parent / "py-utils"
if str(_py_utils) not in sys.path:
    sys.path.insert(0, str(_py_utils))

import pandas as pd

import m_viz_utils
from m_viz_utils import read_matpower_case, summarize_case, validation_report_df

## environment and display setup

Keep the notebook lightweight for now. We only need enough setup to inspect the `.m` case tables cleanly.

In [ ]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

### on kestrel?
print("hello world")
import socket
hostname = socket.gethostname()
# Kestrel compute nodes: x1002c5s3b1n0, x1003c0s2b1n0, etc (always start with 'x' + 4 digits + 'c')
onkestrel = bool(__import__('re').match(r'^x\d{4}c', hostname))

if onkestrel:
    print(f"✓ You are on Kestrel (hostname: {hostname})")
else:
    print(f"✗ You are NOT on Kestrel (hostname: {hostname})")

# configure input paths: select case

Download TAMU synthetic grid cases from
[Texas A&M Electric Grid Test Cases](https://electricgrids.engr.tamu.edu/electric-grid-test-cases/).
After unzipping, point `CASE_DATA_DIR` to the folder containing the extracted files.

Files used (Hawaii40 example):

| File | Used for |
|---|---|
| `Hawaii40_20231026.m` | MATPOWER case (buses, generators, branches) |
| `Hawaii40_GIC_data.gic` | Geographic coordinates (preferred) |
| `Hawaii40_20231026.AUX` | Geographic coordinates (fallback if no `.gic`) |

Files used (ACTIVSg200 / Illinois example):

| File | Used for |
|---|---|
| `case_ACTIVSg200.m` | MATPOWER case (buses, generators, branches) |
| `ACTIVSg200_GIC_data.gic` | Geographic coordinates (preferred) |
| `ACTIVSg200.AUX` | Geographic coordinates (fallback if no `.gic`) |

**To use your own data:** set `CASE_DATA_DIR` to your extracted folder and `CASE_NAME` to match. Everything else is auto-detected.

Optionally, set `GRIDKIT_REPO` to the root of a local GridKit repository clone to augment hover labels with GridKit-assigned bus and branch IDs. JSON case files are read from `GRIDKIT_REPO/examples/PhasorDynamics/`. Set `GRIDKIT_REPO = None` to skip this augmentation.


In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set CASE_NAME, CASE_DATA_DIR, and CASE_JSON_PATH. Everything else is auto-detected.

# CASE_NAME = "Hawaii40"
CASE_NAME = "ACTIVSg200"
# CASE_NAME = "ACTIVSg2000"
# CASE_NAME = "ACTIVSg10k"

# CASE_DATA_DIR: folder containing your extracted .m, .gic, and .AUX files.
# Uncomment and set your own path:
# CASE_DATA_DIR = Path("/path/to/my/ACTIVSg200")

# My paths (Kestrel / Mac) — comment out if setting CASE_DATA_DIR above:
if onkestrel:
    CASE_DATA_DIR = (
        Path("/kfs2/projects/scidac/scidac-data") / CASE_NAME / "raw-tamu-data"
    )
else:
    CASE_DATA_DIR = (
        Path("/Users/isatkaus/projects/scidac/scidac-data")
        / CASE_NAME
        / "raw-tamu-data"
    )

# GRIDKIT_REPO: root of your local GridKit repository clone.
# Optional — set to None to skip GridKit JSON ID augmentation on hover labels.
# JSON case files live inside the GridKit repo under examples/PhasorDynamics/.
GRIDKIT_REPO = Path("/home/isatkaus/gridkit")

_case_json_paths = (
    {
        "Hawaii40": GRIDKIT_REPO / "examples/PhasorDynamics/Medium/Hawaii/hawaii.json",
        "ACTIVSg200": GRIDKIT_REPO
        / "examples/PhasorDynamics/Large/Illinois/illinois.json",
        "ACTIVSg2000": GRIDKIT_REPO / "examples/PhasorDynamics/Large/Texas/texas.json",
        "ACTIVSg10k": GRIDKIT_REPO / "examples/PhasorDynamics/Large/WECC/wecc.json",
    }
    if GRIDKIT_REPO is not None
    else {}
)
CASE_JSON_PATH = _case_json_paths.get(CASE_NAME, None)
# ────────────────────────────────────────────────────────────────────────────

case_name = CASE_NAME
case_data_dir = CASE_DATA_DIR

# File detection: set to None for auto-detect, or specify explicitly.
# Auto-detect looks for case_<name>.m first, then any single .m in case_data_dir.
m_file_name = None
# Hawaii explicit example: m_file_name = "Hawaii40_20231026.m"
gic_file_name = f"{case_name}_GIC_data.gic"
aux_file_name = None  # auto: <m_stem>.AUX/.aux or <case_name>.AUX/.aux

if m_file_name is None:
    default_m = case_data_dir / f"case_{case_name}.m"
    if default_m.exists():
        m_file_path = default_m
    else:
        m_candidates = sorted(case_data_dir.glob("*.m"))
        if len(m_candidates) == 1:
            m_file_path = m_candidates[0]
        else:
            raise FileNotFoundError(
                f"Could not uniquely determine .m file in {case_data_dir}. "
                f"Candidates: {[p.name for p in m_candidates]}"
            )
else:
    m_file_path = case_data_dir / m_file_name

gic_file_path = case_data_dir / gic_file_name

if aux_file_name is None:
    # case-insensitive: try .AUX then .aux
    def _find_aux(stem, directory):
        for ext in (".AUX", ".aux"):
            p = directory / f"{stem}{ext}"
            if p.exists():
                return p
        return directory / f"{stem}.AUX"  # canonical non-existent path for warnings

    default_aux = _find_aux(m_file_path.stem, case_data_dir)
    if default_aux.exists():
        aux_file_path = default_aux
    else:
        aux_file_path = _find_aux(case_name, case_data_dir)
else:
    aux_file_path = case_data_dir / aux_file_name

gic_exists = gic_file_path.exists()
aux_exists = aux_file_path.exists()

geo_file_path = gic_file_path if gic_exists else aux_file_path

if not m_file_path.exists():
    raise FileNotFoundError(f"MATPOWER .m file not found: {m_file_path}")
if not gic_exists:
    if aux_exists:
        print(f"INFO: GIC not found; using AUX for geo: {aux_file_path}")
    else:
        print(f"WARNING: GIC file not found at {gic_file_path}")
        print(f"WARNING: AUX file not found at {aux_file_path}")
        print("WARNING: No geo source file found (.gic or .AUX)")

print(f"case_data_dir:  {case_data_dir}")
print(f"m_file_path:    {m_file_path}")
print(f"gic_file_path:  {gic_file_path}  (exists: {gic_exists})")
print(f"aux_file_path:  {aux_file_path}  (exists: {aux_exists})")
print(f"geo_file_path:  {geo_file_path}")
print(
    f"case_json_path: {CASE_JSON_PATH}  (exists: {CASE_JSON_PATH is not None and CASE_JSON_PATH.exists()})"
)

## load MATPOWER case

Read the `.m` file into normalized `bus` / `gen` / `branch` tables via `m_viz_utils.py`.

In [ ]:
importlib.reload(m_viz_utils)
from m_viz_utils import read_matpower_case, summarize_case, validation_report_df

case_data = read_matpower_case(m_file_path)
summary = summarize_case(case_data)
validation_df = validation_report_df(case_data)

summary

## validation

These checks confirm that generator and branch bus references map back to valid `BUS_I` values.

In [ ]:
validation_df

## quick previews

Preview the normalized tables before adding geo joins or plotting logic.

In [ ]:
pd.set_option("display.max_rows", 6)
print("bus table")
case_data.bus

print("gen table")
case_data.gen

print("branch table")
case_data.branch

## optional auxiliary tables

If the MATPOWER file exposes generator fuel labels or cost tables, inspect them here.

In [ ]:
if case_data.genfuel is not None:
    print("genfuel index")
    case_data.genfuel

if case_data.gencost is not None:
    print("gencost table")
    case_data.gencost.head()

## geo merge and coordinate loading

Loads bus coordinates from `.gic` or `.AUX`, merges into the MATPOWER tables, splits colocated buses, and fans out generators that share a bus.

The geo merge result is passed to the geo plot cell below.


In [ ]:
importlib.reload(m_viz_utils)
import plotly.graph_objects as go
from m_viz_utils import attach_geo_to_case

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Circle radius (degrees lat/lon, a distance) for spreading colocated buses.
# Per-case defaults; override by setting SPLIT_RADIUS to a fixed value instead.
_split_radius_defaults = {
    "Hawaii40": 0.006,
    "ACTIVSg200": 0.006,
    "ACTIVSg2000": 0.02,
    "ACTIVSg10k": 0.04,
}
SPLIT_RADIUS = _split_radius_defaults.get(case_name, 0.006)

# Circle radius (degrees lat/lon, a distance) for spreading colocated buses.
# Per-case defaults; override by setting GEN_FANOUT_RADIUS to a fixed value instead.
_fanout_radius_defaults = {
    "Hawaii40": 0.0025,
    "ACTIVSg200": 0.004,
    "ACTIVSg2000": 0.008,
    "ACTIVSg10k": 0.015,
}
GEN_FANOUT_RADIUS = _fanout_radius_defaults.get(case_name, 0.004)

# FAN_SINGLE_GEN: offset single-generator buses by GEN_FANOUT_RADIUS (eastward)
# so the bus marker underneath remains individually hoverable.
#   False — single-gen buses stay at bus center (default; consistent across cases)
#   True  — single-gen buses also get a small eastward offset
FAN_SINGLE_GEN = False
# ────────────────────────────────────────────────────────────────────────────

# Fanout is enabled automatically when any bus has more than one generator.
n_multi_gen_buses = int((case_data.gen.groupby("GEN_BUS").size() > 1).sum())
enable_gen_fanout = n_multi_gen_buses > 0 or FAN_SINGLE_GEN
split_radius = SPLIT_RADIUS
gen_fanout_radius = GEN_FANOUT_RADIUS
fan_single_gen = FAN_SINGLE_GEN

if geo_file_path.exists():
    geo_result = attach_geo_to_case(
        case_data,
        geo_file_path,
        split_colocated=True,
        split_radius=split_radius,
        gen_fanout=enable_gen_fanout,
        gen_fanout_radius=gen_fanout_radius,
        gen_fanout_singles=fan_single_gen,
    )
    geo_case_data = geo_result.case_data

    # --- optional: attach GridKit JSON ids to hover text ---
    import gridkit_utils

    importlib.reload(gridkit_utils)
    from gridkit_utils import attach_json_ids

    case_json_path = CASE_JSON_PATH
    if case_json_path is not None and case_json_path.exists():
        attach_json_ids(geo_case_data, case_json_path)
        print(f"JSON ids attached from {case_json_path.name}")
    else:
        print("JSON ids not attached (CASE_JSON_PATH is None or file not found)")

    pd.Series(
        {
            "geo_source": str(geo_file_path),
            "split_applied": geo_result.split_applied,
            "split_radius": split_radius,
            "gen_fanout_enabled": enable_gen_fanout,
            "gen_fanout_radius": gen_fanout_radius,
            "fan_single_gen": fan_single_gen,
            "n_buses_with_multiple_gens": n_multi_gen_buses,
            "unique_bus_locations_before": geo_result.n_unique_bus_locations_before,
            "unique_bus_locations_after": geo_result.n_unique_bus_locations_after,
        }
    )
else:
    geo_result = None
    geo_case_data = None
    print(
        "Geo prep skipped: no geo source file found. "
        "Set geo_file_path to a valid .gic or .AUX file, then rerun this cell."
    )

## geo plot

Map with branches, buses (sized by PD), and generators (fuel-colored when available).

Branch **line loading** (%) = max(|S_F|, |S_T|) / RATE_A × 100, where S_F = √(PF²+QF²) and S_T = √(PT²+QT²). Requires PF, QF, PT, QT, and RATE_A columns in the `.m` branch table. Branches with RATE_A = 0 are shown in gray.

### Fault bus selection (optional)

Set `fault_bus` to highlight a bus with a red marker. Bus faults only for now (branch faults not yet supported). Set to None to skip the fault overlay entirely.

`fault_bus` accepts any of four identifiers:

| Identifier | Source | Example (Hawaii40) | Example (ACTIVSg200) |
|---|---|---|---|
| `BUS_I` (int) | `.m` file `mpc.bus` col 1 | `1` | `49` |
| `bus_name` (str) | `.m` file `mpc.bus_name` | `"ALOHA138"` | `"RANTOUL 2 1"` |
| JSON bus number (int) | `"number"` field in JSON `Bus` device | `1` (same as `BUS_I`) | `49` (same as `BUS_I`) |
| JSON bus name (str) | `"name"` field in JSON `Bus` device | `"ALOHA138"` (same as `bus_name`) | `"RANTOUL 2 1"` (same) |

For both Hawaii40 and ACTIVSg200, the JSON `"number"` and `"name"` fields match the `.m` file's `BUS_I` and `bus_name` exactly, so all four forms are equivalent.


In [ ]:
# ============ make general plot ================

importlib.reload(m_viz_utils)
from m_viz_utils import plot_grid, lookup_fault_bus

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# SHOW_LOADING = False  # False: uniform gray branches
SHOW_LOADING = True  # True: viridis-colored branches by loading_pct

# SAVE_HTML = False
SAVE_HTML = True

# SAVE_FIGS_DIR: directory where HTML figures are saved.
# Uncomment and set your own path:
# SAVE_FIGS_DIR = Path("/path/to/my/figs")

# My paths (Kestrel / Mac):
if onkestrel:
    SAVE_FIGS_DIR = Path("/home/isatkaus/gridkit/uq-usecase/figs")
else:
    SAVE_FIGS_DIR = Path("/Users/isatkaus/projects/scidac/scidac-data/figs")

# FIG_WIDTH / FIG_HEIGHT: figure size in pixels. None = plot_grid defaults (width=780, height=650).
FIG_WIDTH = 920
FIG_HEIGHT = None

# MARKER_SCALE: global multiplier for all circle marker sizes (buses and generators).
#   1.0  — default sizes
#   0.5  — half size (useful for large cases with many overlapping markers)
#   2.0  — double size
MARKER_SCALE = 1.0

# ZOOM: map zoom level. Higher = closer; per-case defaults for appropriate centering.
# Typical range: 3 (very far) to 10 (very close). Adjust to fit your grid size.
_zoom_defaults = {
    "Hawaii40": 9,
    "ACTIVSg200": 7,
    "ACTIVSg2000": 4,
    "ACTIVSg10k": 3,
}
ZOOM = _zoom_defaults.get(case_name, 4)

# CENTER: map center (lat, lon). Per-case defaults for proper centering when zoomed in.
# If None, Plotly auto-centers; but explicit centering ensures consistent zoom positioning.
# To auto-calculate: center = (geo_case_data.bus['lat'].mean(), geo_case_data.bus['lon'].mean())
_center_defaults = {
    "Hawaii40": (20.9, -156.5),
    "ACTIVSg200": (40.1, -89.0),
    "ACTIVSg2000": (31.5, -97.5),
    "ACTIVSg10k": (39.0, -107.0),
}
CENTER = _center_defaults.get(case_name, None)

# FAULT_BUS: bus to highlight with a red fault marker. Set to None to skip.
# Bus faults only for now (branch faults not yet supported).
# Accepts any of four identifiers (all equivalent for Hawaii40 and ACTIVSg200):
#   BUS_I (int)     from .m file mpc.bus col 1   e.g. 1          or 49
#   bus_name (str)  from .m file mpc.bus_name     e.g. "ALOHA138" or "RANTOUL 2 1"
#   JSON "number"   Bus device "number" field      e.g. 1          or 49   (= BUS_I)
#   JSON "name"     Bus device "name" field        e.g. "ALOHA138" or "RANTOUL 2 1" (= bus_name)
_fault_defaults = {
    "Hawaii40": "ALOHA138",
    "ACTIVSg200": 147,  # parametric — all 200 buses are wired with BusFault devices
    "ACTIVSg2000": 6349,  ### 6349, 7428, 7422
    "ACTIVSg10k": 100,
}
FAULT_BUS = _fault_defaults.get(case_name, None)

# PLACE_MARKER_AT_BUS: drop a standout locator marker at a bus of interest so you can
# quickly find it on a large grid. Appears in the legend — click the legend entry to
# hide/show it, just like the fault bus marker. No hover tooltip.
# Accepts the same identifiers as FAULT_BUS (BUS_I int, bus_name str, JSON number/name).
# Set to None to skip.
# PLACE_MARKER_AT_BUS = None
PLACE_MARKER_AT_BUS = 161  # e.g. ACTIVSg200: SPRINGFIELD 4 1
# PLACE_MARKER_AT_BUS = "ALOHA138"

# INCLUDE_PLOTLYJS: controls how plotly.js is included in the saved HTML.
#   True    — bundles plotly.js (~3 MB); required for MkDocs iframes and offline viewing (default)
#   "cdn"   — loads plotly.js from CDN at open time; ~50 KB file, but requires internet access
INCLUDE_PLOTLYJS = True
# INCLUDE_PLOTLYJS = "cdn"
# ────────────────────────────────────────────────────────────────────────────

show_loading = SHOW_LOADING
save_html = SAVE_HTML
save_figs_dir = SAVE_FIGS_DIR
fig_width = FIG_WIDTH
fig_height = FIG_HEIGHT
marker_scale = MARKER_SCALE
zoom = ZOOM
center = CENTER
fault_bus = FAULT_BUS

# Show gen connectors whenever gen fanout was enabled in the geo merge cell.
show_gen_connectors = enable_gen_fanout

if geo_case_data is not None:
    # Pass center to plot_grid so it's set correctly from the start
    print(f"Creating plot with zoom={zoom}, center={center}")
    fig_geo = plot_grid(
        geo_case_data,
        zoom=zoom,
        center=center,
        show_loading=show_loading,
        show_gen_connectors=show_gen_connectors,
        marker_scale=marker_scale,
    )

    if fig_width is not None or fig_height is not None:
        _ = fig_geo.update_layout(
            width=fig_width if fig_width is not None else fig_geo.layout.width,
            height=fig_height if fig_height is not None else fig_geo.layout.height,
        )

    # --- standout bus locator marker (legend-toggleable, no hover tooltip) ---
    if PLACE_MARKER_AT_BUS is not None:
        _bus_row = lookup_fault_bus(geo_case_data.bus, PLACE_MARKER_AT_BUS)
        if _bus_row.empty:
            print(f"WARNING: PLACE_MARKER_AT_BUS={PLACE_MARKER_AT_BUS!r} not found.")
        else:
            _lat = float(_bus_row["lat"].iloc[0])
            _lon = float(_bus_row["lon"].iloc[0])
            _label = (
                _bus_row["bus_name"].iloc[0]
                if "bus_name" in _bus_row.columns
                else str(int(_bus_row["BUS_I"].iloc[0]))
            )
            _marker_group = f"marker_{_label}"
            # outer halo (no legend entry)
            _ = fig_geo.add_trace(
                go.Scattermap(
                    mode="markers",
                    lat=[_lat],
                    lon=[_lon],
                    hoverinfo="skip",
                    marker=dict(size=22, color="white", opacity=0.85),
                    legendgroup=_marker_group,
                    showlegend=False,
                )
            )
            # inner dot with label — carries the legend entry
            _ = fig_geo.add_trace(
                go.Scattermap(
                    name=f"📍 marker: {_label}",
                    mode="markers+text",
                    lat=[_lat],
                    lon=[_lon],
                    text=[f"  {_label}"],
                    textposition="middle right",
                    textfont=dict(size=13, color="black"),
                    hoverinfo="skip",
                    marker=dict(size=14, color="gold", opacity=1.0),
                    legendgroup=_marker_group,
                    showlegend=True,
                )
            )
            print(
                f"Standout marker placed at {_label!r} (PLACE_MARKER_AT_BUS={PLACE_MARKER_AT_BUS!r})"
            )

    # --- fault bus overlay ---
    if fault_bus is not None:
        bus_row = lookup_fault_bus(geo_case_data.bus, fault_bus)
        if bus_row.empty:
            bus_df = geo_case_data.bus
            print(
                f"WARNING: fault_bus={fault_bus!r} not found.\n"
                f"Available bus_name values: {bus_df['bus_name'].tolist() if 'bus_name' in bus_df.columns else 'N/A'}\n"
                f"BUS_I range: {bus_df['BUS_I'].min()} – {bus_df['BUS_I'].max()}"
            )
        else:
            fault_lat = float(bus_row["lat"].iloc[0])
            fault_lon = float(bus_row["lon"].iloc[0])
            fault_bus_i = int(bus_row["BUS_I"].iloc[0])
            fault_label = (
                bus_row["bus_name"].iloc[0]
                if "bus_name" in bus_row.columns
                else str(fault_bus_i)
            )
            fault_group = f"fault_{fault_bus_i}"

            _ = fig_geo.add_trace(
                go.Scattermap(
                    mode="markers",
                    lat=[fault_lat],
                    lon=[fault_lon],
                    hoverinfo="skip",
                    marker=dict(size=36, color="black", opacity=1.0),
                    legendgroup=fault_group,
                    showlegend=False,
                )
            )
            _ = fig_geo.add_trace(
                go.Scattermap(
                    name=f"⚡ fault: {fault_label}",
                    mode="markers",
                    lat=[fault_lat],
                    lon=[fault_lon],
                    hovertext=[
                        f"<b>⚡ FAULT</b><br>BUS_I: {fault_bus_i}<br>Name: {fault_label}"
                    ],
                    hoverinfo="text",
                    marker=dict(size=26, color="red", opacity=1.0),
                    legendgroup=fault_group,
                    showlegend=True,
                )
            )

    if save_html:
        save_figs_dir.mkdir(parents=True, exist_ok=True)
        suffix = "_loading" if show_loading else ""
        fault_suffix = f"_fault{fault_bus}" if fault_bus is not None else ""
        html_stem = f"{case_name}_geo{suffix}{fault_suffix}"
        html_path = save_figs_dir / f"{html_stem}.html"
        # height=None + viewport CSS fills browser window in new tab
        fig_html = go.Figure(fig_geo)
        _ = fig_html.update_layout(height=None, width=None, autosize=True)
        _viewport_css = (
            "var s=document.createElement('style');"
            "s.textContent='html,body{height:100vh;margin:0;padding:0;overflow:hidden;}';"
            "document.head.appendChild(s);"
        )
        fig_html.write_html(
            str(html_path),
            full_html=True,
            include_plotlyjs=INCLUDE_PLOTLYJS,
            post_script=_viewport_css,
        )
        print(f"figure saved to {html_path}")

    fig_geo
else:
    print(
        "Geo plot skipped because geo_case_data is not available. "
        "Provide a valid geo_file_path and rerun geo prep cell first."
    )

In [ ]:
#### check file sizes for notebook and saved HTML
def _fmt_mb(path):
    mb = Path(path).stat().st_size / 1024**2
    return f"{mb:.2f} MB  ({path})"


nb_path = Path.cwd() / "m_viz.ipynb"
print(_fmt_mb(nb_path))

if save_html and html_path.exists():
    print(_fmt_mb(html_path))

## plots for paper

Simplified, publication-ready geo map with clean styling:
- All branches and base buses in black/gray (no loading colorbars, no fuel colors)
- Two highlighted bus groups in distinct colors: **measurement buses** (green) and **epistemic generator buses** (blue)
- Optional PNG export (requires `kaleido` library)
- Test run: 4 random buses selected for each group

Modify `MEASUREMENT_BUSES` and `GENERATOR_BUSES` lists to highlight your own bus selections.
Can use either BUS_I (integer) or bus_name (string) identifiers.

**Alternative terminology for paper audience:**
- Epistemic generator buses (parameter uncertainty focus)
- Study generator buses (what they're used for)
- Swept generator buses (parameter sweep in UQ study)

In [ ]:
SAVE_FIGS_DIR_PAPER = (
    "/Users/isatkaus/projects/scidac/repos/gridkit/uq-usecase/work-notes/figures/"
)

In [ ]:
import numpy as np

importlib.reload(m_viz_utils)
from m_viz_utils import plot_grid_paper, parse_json_genrou_names

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Per-case hardcoded bus selections for paper plots

if case_name == "ACTIVSg200":
    # Measurement buses (by .m file bus_name or BUS_I)
    MEASUREMENT_BUSES = ["NORMAL 2 3", "BARTONVILLE 3", "PEKIN 1 2", "CLINTON 1 2"]
    MEASUREMENT_BUSES = []  ### if we want to skip measurement buses

    # Epistemic generator buses (Genrou with inertia parameters varied in UQ study)
    # By JSON Genrou name; we'll extract BUS_ID from "Genrou BUS_ID RANK"
    # Then look up the actual bus names from geo_case_data
    GENROU_GENERATORS = ["Genrou 126 1", "Genrou 135 1", "Genrou 115 1", "Genrou 189 1"]
    GENERATOR_BUSES = parse_json_genrou_names(GENROU_GENERATORS)
else:
    # Fallback for other cases: empty lists
    MEASUREMENT_BUSES = []
    GENERATOR_BUSES = []

# Measurement and generator bus colors
MEASUREMENT_COLOR = "#2ca02c"  # green
GENERATOR_COLOR = "#1f77b4"  # blue

# Fixed marker sizes in pixels (independent of bus load)
# Measurement buses drawn first at larger size (halo); generator buses drawn second at smaller size (on top).
# Adjust these values to control relative visibility when buses coincide.
MEASUREMENT_MARKER_SIZE = 20.0  # pixels (drawn first; larger halo)
GENERATOR_MARKER_SIZE = 20.0  # pixels (drawn second; smaller, on top)
FAULT_MARKER_SIZE = 20.0  # pixels (red fault marker)

# ZOOM and CENTER for paper plot (same as main plot)
ZOOM_PAPER = 7
CENTER_PAPER = (40.1, -89.0)

# FIG_WIDTH / FIG_HEIGHT for paper plot
FIG_WIDTH_PAPER = 920
FIG_HEIGHT_PAPER = 650

# FAULT_BUS: use same defaults as geo plot cell
_fault_defaults = {
    "Hawaii40": "ALOHA138",
    "ACTIVSg200": 66,  # Fault bus for slides
    "ACTIVSg2000": 6349,
    "ACTIVSg10k": 100,
}
FAULT_BUS_PAPER = _fault_defaults.get(case_name, None)

# SAVE_HTML_PAPER: output format
SAVE_HTML_PAPER = True

# SAVE_FIGS_DIR_PAPER: directory for paper figures (same as main plots)
SAVE_FIGS_DIR_PAPER = SAVE_FIGS_DIR
# SAVE_FIGS_DIR_PAPER = Path(str(SAVE_FIGS_DIR) + "-paper")
SAVE_FIGS_DIR_PAPER = SAVE_FIGS_DIR / "paper"  ## subdir option

# Optional legend labels for paper audience — comment/uncomment to toggle terminology
# LEGEND_GENERATOR_LABEL = "epistemic generator buses"  # parameter uncertainty focus (default)
LEGEND_GENERATOR_LABEL = "study generator buses"  # what they're used for
# LEGEND_GENERATOR_LABEL = "swept generator buses"  # parameter sweep in UQ study
# LEGEND_GENERATOR_LABEL = "parameter-varied buses"  # alternative phrasing
# ────────────────────────────────────────────────────────────────────────────

measurement_buses = MEASUREMENT_BUSES
generator_buses = GENERATOR_BUSES
measurement_color = MEASUREMENT_COLOR
generator_color = GENERATOR_COLOR
measurement_marker_size = MEASUREMENT_MARKER_SIZE
generator_marker_size = GENERATOR_MARKER_SIZE
fault_marker_size = FAULT_MARKER_SIZE
zoom_paper = ZOOM_PAPER
center_paper = CENTER_PAPER
fig_width_paper = FIG_WIDTH_PAPER
fig_height_paper = FIG_HEIGHT_PAPER
save_html_paper = SAVE_HTML_PAPER
save_figs_dir_paper = SAVE_FIGS_DIR_PAPER
fault_bus_paper = FAULT_BUS_PAPER
legend_generator_label = LEGEND_GENERATOR_LABEL

if geo_case_data is not None:
    print(f"Paper plot config:")
    print(f"  measurement_buses: {measurement_buses}")
    print(f"  generator_buses (by BUS_ID): {generator_buses}")
    if generator_buses:
        # Look up and display corresponding bus names
        gen_bus_names = []
        for bid in generator_buses:
            row = geo_case_data.bus[geo_case_data.bus["BUS_I"] == bid]
            if not row.empty and "bus_name" in row.columns:
                gen_bus_names.append(row["bus_name"].iloc[0])
            else:
                gen_bus_names.append(f"BUS_{bid}")
        print(f"  generator_buses (bus_names): {gen_bus_names}")
    print(f"  zoom: {zoom_paper}, center: {center_paper}")
    print(
        f"  marker sizes (pixels): measurement={measurement_marker_size} (drawn first), generator={generator_marker_size} (drawn second, on top), fault={fault_marker_size}"
    )
    print(
        f"  drawing order: measurement buses first (larger), generator buses second (smaller, on top)"
    )
    if fault_bus_paper is not None:
        print(f"  fault_bus: {fault_bus_paper} (red marker)")
    print(f"  legend label (generator buses): {legend_generator_label}")

    # Create simplified paper plot
    fig_paper = plot_grid_paper(
        geo_case_data,
        measurement_buses=measurement_buses,
        generator_buses=generator_buses,
        zoom=zoom_paper,
        center=center_paper,
        measurement_color=measurement_color,
        generator_color=generator_color,
        marker_scale=marker_scale,
        measurement_marker_size=measurement_marker_size,
        generator_marker_size=generator_marker_size,
        fault_bus=fault_bus_paper,
        fault_marker_size=fault_marker_size,
        legend_generator_label=legend_generator_label,
    )

    # Adjust dimensions and position legend at top-left corner
    _ = fig_paper.update_layout(
        width=fig_width_paper,
        height=fig_height_paper,
        # legend=dict(x=0, y=1, xanchor="left", yanchor="top"), ## top left corner
        legend=dict(x=1, y=1, xanchor="right", yanchor="top"),  ## top right corner
    )

    # Save HTML
    if save_html_paper:
        save_figs_dir_paper.mkdir(parents=True, exist_ok=True)

        html_stem = f"{case_name}_paper_no_measurement_buses"
        # html_stem = f"{case_name}_paper_20_10"

        html_path_paper = save_figs_dir_paper / f"{html_stem}.html"
        # height=None + width=None + autosize=True enables responsive resizing and PNG download
        fig_paper_html = go.Figure(fig_paper)
        _ = fig_paper_html.update_layout(height=None, width=None, autosize=True)
        _viewport_css = (
            "var s=document.createElement('style');"
            "s.textContent='html,body{height:100vh;margin:0;padding:0;overflow:hidden;}';"
            "document.head.appendChild(s);"
        )
        fig_paper_html.write_html(
            str(html_path_paper),
            full_html=True,
            include_plotlyjs=True,
            post_script=_viewport_css,
        )
        print(f"Paper HTML saved to {html_path_paper}")

    fig_paper
else:
    print("Paper plot skipped: geo_case_data not available.")

# Figures for slides (datacenter report)


In [18]:
########### figures for datacenter-pilot slides

import numpy as np

importlib.reload(m_viz_utils)
from m_viz_utils import (
    plot_grid_datacenter,
    lookup_fault_bus,
    load_hourly_profiles,
    hour_snapshot,
    resolve_hour,
)

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Datacenter slides: topology, loads, IBR (wind) generators in blue, thermal in gray.
# Uses the datacenter-specific plot_grid_datacenter() function.

# Hour for load and wind circles: 0-based index into demand.csv / wind.csv,
# or a DateTime string such as "2017-07-18T22:00:00". None = base case from the .m file.
HOUR_OF_YEAR = 4779
HOUR_OF_YEAR = 477

# Datacenter siting location(s) for reference (optional orange marker overlay)
# Format: (bus_id, size_mw) or just bus_id (defaults to 200 MW)
# Common siting locations: 108 (central), 148 (eastern), 6, 120
_siting_defaults = {
    "Hawaii40": ("ALOHA138", 200),
    "ACTIVSg200": (108, 200),  # Datacenter siting bus with 200 MW size
    "ACTIVSg2000": (6349, 200),
    "ACTIVSg10k": (100, 200),
}
SITING_BUS_SLIDES = _siting_defaults.get(case_name, None)

# Fault bus for slides (optional red marker overlay)
# Format: bus_id (BUS_I int or bus_name str)
# Set to None to skip fault marker
_fault_defaults_slides = {
    "Hawaii40": "ALOHA138",
    "ACTIVSg200": 66,  # Fault bus for slides
    "ACTIVSg2000": 6349,
    "ACTIVSg10k": 100,
}
FAULT_BUS_SLIDES = _fault_defaults_slides.get(case_name, None)

# Figure dimensions for slides
FIG_WIDTH_SLIDES = 920
FIG_HEIGHT_SLIDES = 650

# Zoom and center
ZOOM_SLIDES = 7
CENTER_SLIDES = (40.1, -89.0) if case_name == "ACTIVSg200" else None

# Marker scale (1.0 = default, 0.5 = half size for dense grids)
MARKER_SCALE_SLIDES = 1.0

# Save output
SAVE_HTML_SLIDES = True
if onkestrel:
    SAVE_FIGS_DIR_SLIDES = Path(
        "/kfs2/projects/scidac/scidac-repos/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot"
    )
else:
    SAVE_FIGS_DIR_SLIDES = Path(
        "/Users/isatkaus/projects/scidac/repos/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot"
    )

# Hourly demand/wind profiles sit in the grid-v1 folder next to raw-tamu-data
HOURLY_PROFILE_DIR = CASE_DATA_DIR.parent / "grid-v1"
HOURLY_PROFILES_AVAILABLE = case_name == "ACTIVSg200" and (
    HOURLY_PROFILE_DIR / "demand.csv"
).exists()
# ────────────────────────────────────────────────────────────────────────────

show_loading_slides = False  # No loading coloring for clean slide appearance
siting_bus_slides = SITING_BUS_SLIDES
fault_bus_slides = FAULT_BUS_SLIDES
fig_width_slides = FIG_WIDTH_SLIDES
fig_height_slides = FIG_HEIGHT_SLIDES
zoom_slides = ZOOM_SLIDES
center_slides = CENTER_SLIDES
marker_scale_slides = MARKER_SCALE_SLIDES
save_html_slides = SAVE_HTML_SLIDES
save_figs_dir_slides = SAVE_FIGS_DIR_SLIDES

if geo_case_data is not None:
    plot_case = geo_case_data
    hour_idx = None
    hour_label = "base case"
    if HOUR_OF_YEAR is not None and HOURLY_PROFILES_AVAILABLE:
        hourly_profiles = load_hourly_profiles(HOURLY_PROFILE_DIR)
        hour_idx = resolve_hour(hourly_profiles, HOUR_OF_YEAR)
        plot_case = hour_snapshot(geo_case_data, hourly_profiles, hour_idx)
        hour_stamp = hourly_profiles.timestamps.iloc[hour_idx]
        hour_label = f"hour {hour_idx} ({hour_stamp})"
        print(
            f"Hour {hour_idx} ({hour_stamp}): load {plot_case.bus['PD'].sum():.1f} MW, "
            f"wind {hourly_profiles.wind_mw.iloc[hour_idx].sum():.1f} MW"
        )
    elif HOUR_OF_YEAR is not None:
        print("WARNING: no hourly profiles for this case; plotting base case.")

    print(f"Slide plot config:")
    print(f"  case: {case_name}")
    print(f"  hour: {hour_label}")
    print(f"  zoom: {zoom_slides}, center: {center_slides}")
    print(f"  marker_scale: {marker_scale_slides}")
    print(f"  figure size: {fig_width_slides} × {fig_height_slides} px")
    print(f"  siting_bus (optional marker): {siting_bus_slides}")
    print(f"  save directory: {save_figs_dir_slides}")

    # Create datacenter plot: topology (branches), loads (buses sized by PD),
    # and generators separated only by type: IBR/wind (blue) vs thermal (dark gray)
    fig_slides = plot_grid_datacenter(
        plot_case,
        zoom=zoom_slides,
        center=center_slides,
        show_loading=show_loading_slides,
        show_gen_connectors=enable_gen_fanout,
        marker_scale=marker_scale_slides,
    )

    # Adjust figure dimensions and legend position
    _ = fig_slides.update_layout(
        width=fig_width_slides,
        height=fig_height_slides,
        legend=dict(x=1, y=1, xanchor="right", yanchor="top"),  ## top right corner
    )
    if hour_idx is not None:
        _ = fig_slides.update_layout(title=dict(text=hour_label, x=0.01, font=dict(size=14)))

    # --- optional: overlay a siting bus marker (e.g., datacenter placement location) ---
    if siting_bus_slides is not None:
        # Parse (bus, size_mw) format or just bus
        if isinstance(siting_bus_slides, tuple):
            siting_bus_id, siting_size_mw = siting_bus_slides
        else:
            siting_bus_id, siting_size_mw = siting_bus_slides, 200  # default 200 MW

        bus_row = lookup_fault_bus(geo_case_data.bus, siting_bus_id)
        if bus_row.empty:
            print(f"WARNING: siting_bus={siting_bus_id!r} not found.")
        else:
            siting_lat = float(bus_row["lat"].iloc[0])
            siting_lon = float(bus_row["lon"].iloc[0])
            siting_bus_i = int(bus_row["BUS_I"].iloc[0])
            siting_label = (
                bus_row["bus_name"].iloc[0]
                if "bus_name" in bus_row.columns
                else str(siting_bus_i)
            )
            siting_group = f"siting_{siting_bus_i}"

            # Calculate marker size using same formula as load buses
            # size = (sqrt(MW.clip(0.01)) * 2.2 + 3) * marker_scale
            siting_marker_size = (
                np.sqrt(max(siting_size_mw, 0.01)) * 2.2 + 3
            ) * marker_scale_slides

            # Orange datacenter siting marker (no halo, size scales with MW)
            _ = fig_slides.add_trace(
                go.Scattermap(
                    name=f"📍 Datacenter: Bus {siting_bus_i}",
                    mode="markers",
                    lat=[siting_lat],
                    lon=[siting_lon],
                    hovertext=[
                        f"<b>Datacenter Siting</b><br>Bus: {siting_label} (BUS_I: {siting_bus_i})<br>Size: {siting_size_mw} MW"
                    ],
                    hoverinfo="text",
                    marker=dict(size=siting_marker_size, color="orange", opacity=0.9),
                    legendgroup=siting_group,
                    showlegend=True,
                )
            )
            print(
                f"Siting marker placed at {siting_label!r} (size: {siting_size_mw} MW, marker size: {siting_marker_size:.1f})"
            )

    # --- optional: overlay a red fault bus marker ---
    if FAULT_BUS_SLIDES is not None:
        fault_row = lookup_fault_bus(geo_case_data.bus, FAULT_BUS_SLIDES)
        if fault_row.empty:
            print(f"WARNING: FAULT_BUS={FAULT_BUS_SLIDES!r} not found.")
        else:
            fault_lat = float(fault_row["lat"].iloc[0])
            fault_lon = float(fault_row["lon"].iloc[0])
            fault_bus_i = int(fault_row["BUS_I"].iloc[0])
            fault_label = (
                fault_row["bus_name"].iloc[0]
                if "bus_name" in fault_row.columns
                else str(fault_bus_i)
            )
            fault_group = f"fault_{fault_bus_i}"

            # Red fault marker (no halo, same size as datacenter marker)
            _ = fig_slides.add_trace(
                go.Scattermap(
                    name=f"⚡ Fault: Bus {fault_bus_i}",
                    mode="markers",
                    lat=[fault_lat],
                    lon=[fault_lon],
                    hovertext=[
                        f"<b>⚡ FAULT</b><br>Bus: {fault_label} (BUS_I: {fault_bus_i})"
                    ],
                    hoverinfo="text",
                    marker=dict(size=16, color="red", opacity=0.9),
                    legendgroup=fault_group,
                    showlegend=True,
                )
            )
            print(f"Fault marker placed at {fault_label!r}")

    # Save HTML
    if save_html_slides:
        save_figs_dir_slides.mkdir(parents=True, exist_ok=True)

        html_stem = f"{case_name}_slides_topology_loads_ibr"
        if hour_idx is not None:
            html_stem += f"_h{hour_idx}"
        html_path_slides = save_figs_dir_slides / f"{html_stem}.html"

        fig_slides_html = go.Figure(fig_slides)
        _ = fig_slides_html.update_layout(height=None, width=None, autosize=True)
        _viewport_css = (
            "var s=document.createElement('style');"
            "s.textContent='html,body{height:100vh;margin:0;padding:0;overflow:hidden;}';"
            "document.head.appendChild(s);"
        )
        fig_slides_html.write_html(
            str(html_path_slides),
            full_html=True,
            include_plotlyjs=True,
            post_script=_viewport_css,
        )
        print(f"Slide figure saved to {html_path_slides}")

    fig_slides
else:
    print("Slide plot skipped: geo_case_data not available.")


<module 'm_viz_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/m_viz_utils.py'>

Hour 477 (2017-01-20T15:00:00): load 1593.9 MW, wind 1.3 MW
Slide plot config:
  case: ACTIVSg200
  hour: hour 477 (2017-01-20T15:00:00)
  zoom: 7, center: (40.1, -89.0)
  marker_scale: 1.0
  figure size: 920 × 650 px
  siting_bus (optional marker): (108, 200)
  save directory: /kfs2/projects/scidac/scidac-repos/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot
Siting marker placed at 'PRINCEVILLE 0' (size: 200 MW, marker size: 34.1)
Fault marker placed at 'MOUNT ZION 0'
Slide figure saved to /kfs2/projects/scidac/scidac-repos/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot/ACTIVSg200_slides_topology_loads_ibr_h477.html


In [45]:
##### animate aleatoric 1

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from m_viz_utils import (
    plot_grid_datacenter,
    lookup_fault_bus,
    load_hourly_profiles,
    hour_snapshot,
    apply_dispatch,
)

# campaign to animate: uncomment one
#ALEA_DIR = Path("/kfs2/projects/scidac/scidac-data/gridkit-campaigns/aleatoric1_ap08_qlim")
ALEA_DIR = Path("/kfs2/projects/scidac/scidac-data/gridkit-campaigns/aleatoric2_ap08_qlim")
# ALEA_DIR = Path("/kfs2/projects/scidac/scidac-data/gridkit-campaigns/<new_campaign>")
ANIM_TAG = ALEA_DIR.name

SHOW_DISPATCH = True    # True: thermal PG and commitment follow the UC dispatch; False: base case
ANIM_N_PICK = "min"     # "min" or "max": n_commit used when a campaign has several per hour
# ANIM_N_PICK = "max"
OFFLINE_GEN_SIZE = 0    # marker size (px) for offline thermal gens; 0 hides them

ANIM_DC_BUS = 108
ANIM_DC_MW = 200
ANIM_FAULT_BUS = 66
FRAME_MS = 800

conds = pd.read_csv(ALEA_DIR / "conditions.csv")
sel = (
    (conds.dc_bus == ANIM_DC_BUS)
    & (conds.dc_mw == ANIM_DC_MW)
    & (conds.fault_bus == ANIM_FAULT_BUS)
)
cands = conds.loc[sel].assign(op=lambda d: d["tag"].str.split("__").str[0])
cands = cands.sort_values(["hour", "n_commit"], ascending=[True, ANIM_N_PICK == "min"])

# one operating point per hour: first candidate with a dispatch file that is AC-settled
dispatch_by_hour = {}
for r in cands.itertuples():
    h_r = int(r.hour)
    if h_r in dispatch_by_hour:
        continue
    f = ALEA_DIR / "dispatch" / f"{r.op}.parquet"
    if not f.exists():
        continue
    disp = pd.read_parquet(f)
    if bool(disp["ac_solved"].all()):
        dispatch_by_hour[h_r] = (int(r.n_commit), disp)
anim_hours = sorted(dispatch_by_hour)
print(f"{len(anim_hours)} hours at bus {ANIM_DC_BUS}, {ANIM_DC_MW} MW, fault {ANIM_FAULT_BUS} ({ANIM_TAG}, n_commit={ANIM_N_PICK}, dispatch={SHOW_DISPATCH})")

anim_profiles = load_hourly_profiles(HOURLY_PROFILE_DIR)


def anim_label(h):
    """Frame title: hour and timestamp; with dispatch on, also committed thermal units and MW."""
    text = f"hour {h} ({anim_profiles.timestamps.iloc[h]}), n_commit {dispatch_by_hour[h][0]}"
    if not SHOW_DISPATCH:
        return text
    disp = dispatch_by_hour[h][1]
    thermal = disp[disp["fuel"] != "wind"]
    on = thermal[thermal["status_ac"] == 1]
    return f"{text} | committed {len(on)}/{len(thermal)} thermal | {on['pg_ac'].sum():.0f} MW thermal"


frame_figs = {}
for h in anim_hours:
    snap = hour_snapshot(geo_case_data, anim_profiles, h)
    if SHOW_DISPATCH:
        snap = apply_dispatch(snap, dispatch_by_hour[h][1])
    frame_figs[h] = plot_grid_datacenter(
        snap,
        zoom=ZOOM_SLIDES,
        center=CENTER_SLIDES,
        marker_scale=MARKER_SCALE_SLIDES,
        offline_size=OFFLINE_GEN_SIZE if SHOW_DISPATCH else None,
    )

ANIM_TRACES = ("Loads (PD)", "IBR (Wind)") + (("Thermal Gens",) if SHOW_DISPATCH else ())
base = frame_figs[anim_hours[0]]
trace_idx = {t.name: i for i, t in enumerate(base.data) if t.name in ANIM_TRACES}

frames = []
for h in anim_hours:
    fh = frame_figs[h]
    frames.append(
        go.Frame(
            name=str(h),
            data=[next(t for t in fh.data if t.name == n) for n in trace_idx],
            traces=list(trace_idx.values()),
            layout=dict(title=dict(text=anim_label(h))),
        )
    )

for bus_id, name, color, size in [
    (ANIM_DC_BUS, f"Datacenter {ANIM_DC_MW} MW: bus {ANIM_DC_BUS}", "orange",
     (np.sqrt(ANIM_DC_MW) * 2.2 + 3) * MARKER_SCALE_SLIDES),
    (ANIM_FAULT_BUS, f"Fault: bus {ANIM_FAULT_BUS}", "red", 16),
]:
    row = lookup_fault_bus(geo_case_data.bus, bus_id)
    if row.empty:
        raise ValueError(f"bus {bus_id} not found")
    _ = base.add_trace(
        go.Scattermap(
            name=name,
            mode="markers",
            lat=[float(row["lat"].iloc[0])],
            lon=[float(row["lon"].iloc[0])],
            hovertext=[name],
            hoverinfo="text",
            marker=dict(size=size, color=color, opacity=0.9),
        )
    )

steps = [
    dict(
        method="animate",
        label=f"{h} {anim_profiles.timestamps.iloc[h]}",
        args=[[str(h)], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
    )
    for h in anim_hours
]
_ = base.update_layout(
    title=dict(
        text=anim_label(anim_hours[0]),
        x=0.01,
        font=dict(size=14),
    ),
    width=FIG_WIDTH_SLIDES,
    height=FIG_HEIGHT_SLIDES,
    sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix="hour: "), x=0.02, len=0.96, pad=dict(t=40))],
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            x=0.02,
            y=0.0,
            xanchor="left",
            yanchor="top",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[None, dict(frame=dict(duration=FRAME_MS, redraw=True), fromcurrent=True, transition=dict(duration=0))],
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False), transition=dict(duration=0))],
                ),
            ],
        )
    ],
)
base.frames = frames

SAVE_FIGS_DIR_SLIDES.mkdir(parents=True, exist_ok=True)
anim_path = SAVE_FIGS_DIR_SLIDES / f"{case_name}_animate_{ANIM_TAG}_bus{ANIM_DC_BUS}_{ANIM_DC_MW}MW_fault{ANIM_FAULT_BUS}.html"
base.write_html(str(anim_path), include_plotlyjs=True, full_html=True)
print(f"Animation saved to {anim_path}")
base


58 hours at bus 108, 200 MW, fault 66 (aleatoric2_ap08_qlim, n_commit=min, dispatch=True)
Animation saved to /kfs2/projects/scidac/scidac-repos/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot/ACTIVSg200_animate_aleatoric2_ap08_qlim_bus108_200MW_fault66.html


In [46]:
##### export animation frames as PNG (campaign set in the animation cell)

ANIM_PNG_DIR = Path("/home/isatkaus/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot/animation")
ANIM_PNG_DIR.mkdir(parents=True, exist_ok=True)

base_data = [t.to_plotly_json() for t in base.data]
# drop slider and play/pause controls so they are not baked into the static PNGs
base_layout = {k: v for k, v in base.layout.to_plotly_json().items() if k not in ("sliders", "updatemenus")}
png_paths = []
for k, h in enumerate(anim_hours):
    data = list(base_data)
    for idx, tr in zip(trace_idx.values(), frames[k].data):
        data[idx] = tr.to_plotly_json()
    fig_k = go.Figure(data=data, layout=base_layout)
    _ = fig_k.update_layout(
        title=dict(text=anim_label(h), x=0.01, font=dict(size=14)),
    )
    assert not fig_k.layout.sliders and not fig_k.layout.updatemenus, "controls leaked into export figure"
    png_path = ANIM_PNG_DIR / f"{case_name}_{ANIM_TAG}_bus{ANIM_DC_BUS}_{ANIM_DC_MW}MW_fault{ANIM_FAULT_BUS}_h{h:04d}.png"
    fig_k.write_image(str(png_path), width=FIG_WIDTH_SLIDES, height=FIG_HEIGHT_SLIDES, scale=2)
    png_paths.append(png_path)
print(f"{len(png_paths)} PNG frames saved to {ANIM_PNG_DIR}")




58 PNG frames saved to /home/isatkaus/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot/animation


In [47]:
##### build MP4 from the PNG frames of the campaign set in the animation cell

import subprocess
import imageio_ffmpeg

ANIM_MP4_PATH = ANIM_PNG_DIR / f"{case_name}_{ANIM_TAG}_bus{ANIM_DC_BUS}_{ANIM_DC_MW}MW_fault{ANIM_FAULT_BUS}.mp4"
frame_list_path = ANIM_PNG_DIR / "frames_concat.txt"
frame_dur = FRAME_MS / 1000

assert all(ANIM_TAG in p.name for p in png_paths), "run the PNG export cell for this campaign first"
lines = []
for p in png_paths:
    lines.append(f"file '{p.resolve()}'")
    lines.append(f"duration {frame_dur}")
# concat demuxer ignores the duration of the last file unless it is repeated
lines.append(f"file '{png_paths[-1].resolve()}'")
_ = frame_list_path.write_text("\n".join(lines) + "\n")

cmd = [
    imageio_ffmpeg.get_ffmpeg_exe(), "-y",
    "-f", "concat", "-safe", "0", "-i", str(frame_list_path),
    "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2,format=yuv420p",
    "-r", "5",
    "-c:v", "libx264", "-movflags", "+faststart",
    str(ANIM_MP4_PATH),
]
res = subprocess.run(cmd, capture_output=True, text=True)
frame_list_path.unlink(missing_ok=True)
if res.returncode != 0:
    raise RuntimeError(res.stderr[-2000:])
print(f"MP4 saved to {ANIM_MP4_PATH}")



MP4 saved to /home/isatkaus/gridkit/uq-usecase/work-notes/tex-report/uq-subsections/figs/uq-pilot/animation/ACTIVSg200_aleatoric2_ap08_qlim_bus108_200MW_fault66.mp4


In [ ]:
# # Verify: Show which measurement and generator buses coincide
# if geo_case_data is not None:
#     print("=" * 80)
#     print("VERIFICATION: Bus overlap between measurement and generator groups")
#     print("=" * 80)

#     # Get the BUS_I values for each group
#     meas_df = geo_case_data.bus[
#         (geo_case_data.bus["bus_name"].isin(MEASUREMENT_BUSES))
#         | (geo_case_data.bus["BUS_I"].isin(MEASUREMENT_BUSES))
#     ]
#     gen_df = geo_case_data.bus[
#         (geo_case_data.bus["bus_name"].isin(GENERATOR_BUSES))
#         | (geo_case_data.bus["BUS_I"].isin(GENERATOR_BUSES))
#     ]

#     meas_bus_ids = set(meas_df["BUS_I"].values)
#     gen_bus_ids = set(gen_df["BUS_I"].values)

#     print(f"\nMeasurement buses (BUS_I): {sorted(meas_bus_ids)}")
#     print(f"Generator buses (BUS_I):   {sorted(gen_bus_ids)}")

#     # Check overlap
#     overlap = meas_bus_ids & gen_bus_ids
#     only_meas = meas_bus_ids - gen_bus_ids
#     only_gen = gen_bus_ids - meas_bus_ids

#     if overlap:
#         print(f"\n✓ OVERLAPPING buses: {sorted(overlap)}")
#         overlap_df = geo_case_data.bus[geo_case_data.bus["BUS_I"].isin(overlap)][
#             ["BUS_I", "bus_name", "lat", "lon"]
#         ]
#         print(overlap_df.to_string(index=False))
#         print(f"\n   Marker visualization (when buses coincide):")
#         print(f"   - Measurement buses: LARGER green circles (drawn first as halo)")
#         print(f"   - Generator buses: SMALLER blue circles (drawn second, on top)")
#         print(f"   Result: BLUE circle visible on top of GREEN circle")
#     else:
#         print(
#             f"\n✗ No overlapping buses — measurement and generator groups are separate."
#         )

#     if only_meas:
#         print(f"\n• Measurement-only buses (large green circles): {sorted(only_meas)}")
#     if only_gen:
#         print(f"• Generator-only buses (small blue circles): {sorted(only_gen)}")

#     print("\n" + "=" * 80)

In [ ]:
# importlib.reload(m_viz_utils)
# from m_viz_utils import plot_grid, lookup_fault_bus

# # --- plot options ---
# show_loading = (
#     True  # True: viridis-colored branches by loading_pct (requires PF/QF/PT/QT/RATE_A)
# )
# # show_loading = False  # False: uniform gray branches

# # save_html = False  # True: write figure to save_figs_dir as .html
# save_html = True

# save_figs_dir = scidac_data_dir / "figs"  # adjust as needed
# # save_figs_dir = Path("/home/isatkaus/projects/scidac/isatkaus/scidac-notebooks/figs")

# show_gen_connectors = "hawaii" in str(case_name).lower()

# # --- fault bus (optional) ---
# # Identify a faulted bus by BUS_I (int) or bus_name (str, from mpc.bus_name).
# # Set to None to skip.
# fault_bus = None
# # fault_bus = 1           # ACTIVSg200: by BUS_I integer
# fault_bus = "ALOHA138"  # Hawaii40: by bus_name string

# if geo_case_data is not None:
#     fig_geo = plot_grid(
#         geo_case_data,
#         zoom=7,
#         show_loading=show_loading,
#         show_gen_connectors=show_gen_connectors,
#     )

#     # --- fault bus overlay ---
#     if fault_bus is not None:
#         bus_row = lookup_fault_bus(geo_case_data.bus, fault_bus)
#         if bus_row.empty:
#             bus_df = geo_case_data.bus
#             print(
#                 f"WARNING: fault_bus={fault_bus!r} not found.\n"
#                 f"Available bus_name values: {bus_df['bus_name'].tolist() if 'bus_name' in bus_df.columns else 'N/A'}\n"
#                 f"BUS_I range: {bus_df['BUS_I'].min()} – {bus_df['BUS_I'].max()}"
#             )
#         else:
#             fault_lat = float(bus_row["lat"].iloc[0])
#             fault_lon = float(bus_row["lon"].iloc[0])
#             fault_bus_i = int(bus_row["BUS_I"].iloc[0])
#             fault_label = (
#                 bus_row["bus_name"].iloc[0]
#                 if "bus_name" in bus_row.columns
#                 else str(fault_bus_i)
#             )
#             fault_group = f"fault_{fault_bus_i}"

#             _ = fig_geo.add_trace(
#                 go.Scattermap(
#                     mode="markers",
#                     lat=[fault_lat],
#                     lon=[fault_lon],
#                     hoverinfo="skip",
#                     marker=dict(size=36, color="black", opacity=1.0),
#                     legendgroup=fault_group,
#                     showlegend=False,
#                 )
#             )
#             _ = fig_geo.add_trace(
#                 go.Scattermap(
#                     name=f"⚡ fault: {fault_label}",
#                     mode="markers",
#                     lat=[fault_lat],
#                     lon=[fault_lon],
#                     hovertext=[
#                         f"<b>⚡ FAULT</b><br>BUS_I: {fault_bus_i}<br>Name: {fault_label}"
#                     ],
#                     hoverinfo="text",
#                     marker=dict(size=26, color="red", opacity=1.0),
#                     legendgroup=fault_group,
#                     showlegend=True,
#                 )
#             )

#     if save_html:
#         save_figs_dir.mkdir(parents=True, exist_ok=True)
#         suffix = "_loading" if show_loading else ""
#         fault_suffix = f"_fault{fault_bus}" if fault_bus is not None else ""
#         html_stem = f"{case_name}_geo{suffix}{fault_suffix}"
#         html_path = save_figs_dir / f"{html_stem}.html"
#         # height=None → 100% of container; fills browser window in new tab,
#         # fits the iframe height in the MkDocs page without internal scrollbars
#         fig_html = go.Figure(fig_geo)
#         _ = fig_html.update_layout(height=None)
#         fig_html.write_html(str(html_path), full_html=True, include_plotlyjs=True)
#         print(f"figure saved to {html_path}")

#     fig_geo
# else:
#     print(
#         "Geo plot skipped because geo_case_data is not available. "
#         "Provide a valid geo_file_path and rerun geo prep cell first."
#     )

In [ ]:
show_gen_connectors = "hawaii" in str(case_name).lower()

# --- figure size (pixels; None = use plot_grid defaults: width=780, height=650) ---
fig_width = None
fig_height = None